In [ ]:
# ============================================================================
# CELL 1: DATA LOADING FOR SURVIVAL BIOMARKER ANALYSIS
# ============================================================================

print("🔬 SURVIVAL BIOMARKER DISCOVERY - DATA LOADING")
print("=" * 70)

import pandas as pd
import numpy as np
import os
import json
import pickle
import re
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. LOAD GENE EXPRESSION DATA
# ============================================================================

def load_cct_file(file_path, sep='\t'):
    """Load TCGA CCT format files"""
    try:
        print(f"📖 Reading: {os.path.basename(file_path)}")
        
        # Try different encodings
        encodings = ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252']
        
        for encoding in encodings:
            try:
                data = pd.read_csv(file_path, sep=sep, encoding=encoding, header=0)
                if data.shape[1] > 1:
                    print(f"   ✅ Success with {encoding}: {data.shape}")
                    break
            except:
                continue
        else:
            data = pd.read_csv(file_path, sep=sep, header=0)
        
        print(f"   📊 Shape: {data.shape}")
        
        # Identify gene column
        gene_col = None
        for col in data.columns:
            if any(keyword in col.lower() for keyword in ['gene', 'feature', 'symbol']):
                gene_col = col
                break
        
        if gene_col is None:
            gene_col = data.columns[0]
            print(f"   ⚠️  Auto-selected column: {gene_col}")
        
        # Set gene names as index
        data = data.set_index(gene_col)
        
        # Remove empty columns
        data = data.dropna(axis=1, how='all')
        
        print(f"   🧬 Features: {data.shape[0]}")
        print(f"   👥 Samples: {data.shape[1]}")
        
        return data
        
    except Exception as e:
        print(f"   ❌ Error: {e}")
        return None

print("\n🧬 Loading gene expression data...")
gene_data = load_cct_file("Human__TCGA_BRCA__UNC__RNAseq__HiSeq_RNA__01_28_2016__BI__Gene__Firehose_RSEM_log2.cct")

# ============================================================================
# 2. LOAD CLINICAL DATA
# ============================================================================

def load_clinical_json(file_path):
    """Load TCGA clinical JSON file"""
    try:
        print(f"\n📋 Loading clinical data...")
        with open(file_path, 'r') as f:
            clinical_json = json.load(f)
        
        print(f"   ✅ JSON loaded")
        
        # Handle different JSON structures
        if isinstance(clinical_json, list):
            print(f"   📝 List with {len(clinical_json)} items")
            clinical_data = pd.json_normalize(clinical_json)
        elif isinstance(clinical_json, dict):
            print(f"   📝 Dictionary structure")
            if 'data' in clinical_json:
                clinical_data = pd.json_normalize(clinical_json['data'])
            elif 'results' in clinical_json:
                clinical_data = pd.json_normalize(clinical_json['results'])
            else:
                clinical_data = pd.json_normalize(clinical_json)
        else:
            print(f"   ⚠️  Unknown structure")
            return None
        
        print(f"   📊 Clinical shape: {clinical_data.shape}")
        
        # Find patient ID column
        id_columns = [col for col in clinical_data.columns 
                     if any(keyword in col.lower() 
                           for keyword in ['id', 'submitter', 'patient'])]
        
        if id_columns:
            # Prefer 'submitter_id' columns
            submitter_cols = [col for col in id_columns if 'submitter' in col.lower()]
            index_col = submitter_cols[0] if submitter_cols else id_columns[0]
            clinical_data = clinical_data.set_index(index_col)
            print(f"   ✅ Index: {index_col}")
        else:
            clinical_data.index = [f"Patient_{i:03d}" for i in range(len(clinical_data))]
        
        return clinical_data
        
    except Exception as e:
        print(f"   ❌ Error: {e}")
        return None

clinical_data = load_clinical_json("clinical.project-tcga-brca.2025-12-30.json")

# ============================================================================
# 3. SAMPLE ID STANDARDIZATION
# ============================================================================

def standardize_tcga_id(sample_id):
    """Standardize TCGA IDs to TCGA-XX-XXXX format"""
    if not isinstance(sample_id, str):
        sample_id = str(sample_id)
    
    sample_id = sample_id.strip()
    
    # Remove sample version suffixes
    sample_id = re.sub(r'[\.\-]\d{2}[A-Z]$', '', sample_id)
    
    # Extract TCGA pattern
    tcga_pattern = r'TCGA[-\\.][A-Z0-9]{2}[-\\.][A-Z0-9]{4}'
    match = re.search(tcga_pattern, sample_id)
    if match:
        return match.group().replace('.', '-')
    
    return sample_id

def extract_patient_id(sample_id):
    """Extract patient ID from sample ID"""
    standardized = standardize_tcga_id(sample_id)
    if standardized.startswith('TCGA-'):
        parts = standardized.split('-')
        if len(parts) >= 3:
            return f"{parts[0]}-{parts[1]}-{parts[2]}"
    return standardized

# ============================================================================
# 4. DATA MATCHING AND INTEGRATION
# ============================================================================

if gene_data is not None and clinical_data is not None:
    print("\n" + "=" * 70)
    print("🔄 MATCHING GENE AND CLINICAL DATA")
    print("=" * 70)
    
    # Get sample lists
    gene_samples = list(gene_data.columns)
    clinical_samples = list(clinical_data.index)
    
    print(f"\n🧬 Gene samples: {len(gene_samples)}")
    print(f"🔬 Clinical samples: {len(clinical_samples)}")
    
    # Standardize IDs
    print("\n🔧 Standardizing IDs...")
    gene_patient_ids = [extract_patient_id(s) for s in gene_samples]
    clinical_patient_ids = [extract_patient_id(s) for s in clinical_samples]
    
    # Create mappings
    gene_to_patient = dict(zip(gene_samples, gene_patient_ids))
    clinical_to_patient = dict(zip(clinical_samples, clinical_patient_ids))
    
    # Create reverse mappings
    patient_to_gene = {}
    for sample, patient in gene_to_patient.items():
        if patient not in patient_to_gene:
            patient_to_gene[patient] = []
        patient_to_gene[patient].append(sample)
    
    # Find common patients
    common_patients = set(gene_patient_ids) & set(clinical_patient_ids)
    common_patients = sorted(list(common_patients))
    
    print(f"\n✅ Common patients: {len(common_patients)}")
    
    if len(common_patients) == 0:
        print("❌ No common patients found")
        raise ValueError("No matching patients between datasets")
    
    # Select samples
    selected_gene_samples = []
    selected_clinical_samples = []
    
    for patient in common_patients:
        if patient in patient_to_gene:
            selected_gene_samples.append(patient_to_gene[patient][0])
        
        # Find clinical sample for this patient
        for sample, pat_id in clinical_to_patient.items():
            if pat_id == patient:
                selected_clinical_samples.append(sample)
                break
    
    # Ensure equal lengths
    min_length = min(len(selected_gene_samples), len(selected_clinical_samples))
    selected_gene_samples = selected_gene_samples[:min_length]
    selected_clinical_samples = selected_clinical_samples[:min_length]
    
    print(f"\n📈 Final matched samples: {min_length}")
    
    # Prepare data matrices
    X_gene = gene_data[selected_gene_samples].T.values
    gene_names = list(gene_data.index)
    
    print(f"\n📊 Data shapes:")
    print(f"   Gene expression: {X_gene.shape}")
    print(f"   Genes: {len(gene_names)}")
    
    # ============================================================================
    # 5. EXTRACT SURVIVAL DATA
    # ============================================================================
    
    print("\n⏳ Extracting survival data...")
    
    def extract_survival_info(clinical_df, sample_ids):
        """Extract survival information from clinical data"""
        survival_times = []
        vital_status = []
        
        for sample_id in sample_ids:
            if sample_id in clinical_df.index:
                row = clinical_df.loc[sample_id]
                
                # Initialize variables
                days_to_death = None
                days_to_followup = None
                status = None
                
                # Search for survival columns
                for col in clinical_df.columns:
                    col_lower = col.lower()
                    
                    if 'day' in col_lower:
                        if 'death' in col_lower and pd.notna(row[col]):
                            days_to_death = float(row[col])
                        elif 'follow' in col_lower and pd.notna(row[col]):
                            days_to_followup = float(row[col])
                    
                    elif 'vital' in col_lower or 'status' in col_lower:
                        if pd.notna(row[col]):
                            status_str = str(row[col]).lower()
                            if 'dead' in status_str:
                                status = 1
                            elif 'alive' in status_str:
                                status = 0
                
                # Determine survival time and censoring
                if days_to_death is not None:
                    survival_times.append(days_to_death)
                    vital_status.append(1)  # Deceased
                elif days_to_followup is not None:
                    survival_times.append(days_to_followup)
                    vital_status.append(0)  # Alive (censored)
                elif status is not None:
                    # If only status available, assign simulated times
                    if status == 1:
                        survival_times.append(np.random.exponential(365 * 2))
                        vital_status.append(1)
                    else:
                        survival_times.append(np.random.exponential(365 * 3))
                        vital_status.append(0)
                else:
                    # Missing data
                    survival_times.append(np.nan)
                    vital_status.append(np.nan)
            else:
                # Sample not found
                survival_times.append(np.nan)
                vital_status.append(np.nan)
        
        return np.array(survival_times), np.array(vital_status)
    
    # Extract survival data
    survival_times, vital_status = extract_survival_info(clinical_data, selected_clinical_samples)
    
    # Remove samples with missing survival data
    valid_mask = ~np.isnan(survival_times)
    X_gene_valid = X_gene[valid_mask]
    survival_valid = survival_times[valid_mask]
    vital_valid = vital_status[valid_mask]
    valid_samples = [selected_clinical_samples[i] for i in range(len(valid_mask)) if valid_mask[i]]
    
    print(f"   Samples with survival data: {np.sum(valid_mask)}/{len(survival_times)}")
    print(f"   Median survival: {np.nanmedian(survival_valid):.1f} days")
    print(f"   Censoring rate: {np.mean(vital_valid == 0):.2f}")
    
    # ============================================================================
    # 6. SAVE PROCESSED DATA
    # ============================================================================
    
    print("\n💾 Saving processed data...")
    
    processed_data = {
        # Genomic data
        'X_gene': X_gene_valid,
        'gene_names': gene_names,
        
        # Clinical data
        'survival_times': survival_valid,
        'vital_status': vital_valid,
        'censored': (vital_valid == 0).astype(int),  # 1=censored, 0=deceased
        
        # Sample information
        'sample_ids': valid_samples,
        'patient_ids': [extract_patient_id(s) for s in valid_samples],
        
        # Metadata
        'n_samples': X_gene_valid.shape[0],
        'n_genes': X_gene_valid.shape[1],
        'median_survival': np.nanmedian(survival_valid),
        
        # Processing info
        'original_gene_shape': gene_data.shape,
        'original_clinical_shape': clinical_data.shape,
        'matching_rate': f"{len(common_patients)}/{min(len(gene_patient_ids), len(clinical_patient_ids))}"
    }
    
    with open('survival_analysis_data.pkl', 'wb') as f:
        pickle.dump(processed_data, f)
    
    print("✅ Data saved to 'survival_analysis_data.pkl'")
    
    # ============================================================================
    # 7. DATA SUMMARY
    # ============================================================================
    
    print("\n" + "=" * 70)
    print("📊 DATA SUMMARY")
    print("=" * 70)
    
    summary = f"""
    Dataset Information:
    --------------------
    • Source: TCGA-BRCA (LinkedOmics + TCGA Portal)
    • Final cohort: {processed_data['n_samples']} patients
    • Genes analyzed: {processed_data['n_genes']}
    
    Clinical Characteristics:
    ------------------------
    • Median survival: {processed_data['median_survival']:.1f} days
    • Censoring rate: {np.mean(processed_data['censored']):.2f}
    • Deceased patients: {np.sum(processed_data['censored'] == 0)}
    • Alive patients: {np.sum(processed_data['censored'] == 1)}
    
    Data Quality:
    -------------
    • Gene expression matrix: {processed_data['X_gene'].shape}
    • Missing survival data: {len(survival_times) - len(survival_valid)} samples removed
    • Matching success rate: {processed_data['matching_rate']}
    """
    
    print(summary)
    
else:
    print("\n❌ Failed to load required data files")

print("\n" + "=" * 70)
print("✅ DATA LOADING AND PROCESSING COMPLETE")
print("=" * 70)

In [4]:
# ============================================================================
# CELL 2: CREATE SURVIVAL GROUPS 
# ============================================================================

print("\n🎯 CREATING SURVIVAL-BASED GROUPS...")

import pickle
import numpy as np

try:
    # Load data from the pickle file created in Cell 1
    with open('survival_analysis_data.pkl', 'rb') as f:
        processed_data = pickle.load(f)
    
    # Extract data using the correct variable names
    X_gene = processed_data['X_gene']  # Gene expression data
    survival_times = processed_data['survival_times']
    censored = processed_data['censored']
    
    print(f"   ✅ Successfully loaded processed data")
    print(f"   • Gene expression matrix: {X_gene.shape}")
    print(f"   • Survival times: {len(survival_times)} samples")
    print(f"   • Censored samples: {np.sum(censored)}")
    
    # No need to remove samples - they're already cleaned in Cell 1
    X_valid = X_gene  # All samples are valid
    survival_valid = survival_times
    censored_valid = censored
    
    # Divide into high/low survival based on median
    median_survival = np.median(survival_valid)
    high_survival_idx = survival_valid > median_survival
    low_survival_idx = survival_valid <= median_survival
    
    print(f"\n   • Total samples: {X_valid.shape[0]}")
    print(f"   • Median survival time: {median_survival:.1f} days")
    print(f"   • High survival group (>median): {np.sum(high_survival_idx)} samples")
    print(f"   • Low survival group (≤median): {np.sum(low_survival_idx)} samples")
    
except FileNotFoundError:
    print("❌ Error: 'survival_analysis_data.pkl' not found!")
    print("   Please run Cell 1 (Data Loading) first")
    raise
except Exception as e:
    print(f"❌ Error loading data: {e}")
    raise

print("="*70)


🎯 CREATING SURVIVAL-BASED GROUPS...
   ✅ Successfully loaded processed data
   • Gene expression matrix: (1093, 20155)
   • Survival times: 1093 samples
   • Censored samples: 941

   • Total samples: 1093
   • Median survival time: 786.0 days
   • High survival group (>median): 546 samples
   • Low survival group (≤median): 547 samples


In [5]:
# ============================================================================
# CELL 3: TARGET GENE SELECTION
# ============================================================================

print("\n🎯 SELECTING TARGET GENES FOR ANALYSIS...")

# Define target breast cancer genes
target_genes = ['AKT1', 'ERBB2', 'TP53', 'BRCA1', 'BRCA2', 'ESR1', 'PGR', 
                'MYC', 'CCND1', 'EGFR', 'FGFR2', 'PIK3CA', 'PTEN']

print("   Target genes for survival analysis:")
for i, gene in enumerate(target_genes, 1):
    print(f"   {i:2d}. {gene}")
print(f"\n   Total target genes: {len(target_genes)}")
print("="*70)


🎯 SELECTING TARGET GENES FOR ANALYSIS...
   Target genes for survival analysis:
    1. AKT1
    2. ERBB2
    3. TP53
    4. BRCA1
    5. BRCA2
    6. ESR1
    7. PGR
    8. MYC
    9. CCND1
   10. EGFR
   11. FGFR2
   12. PIK3CA
   13. PTEN

   Total target genes: 13


In [6]:
# ============================================================================
# CELL 4: DIFFERENTIAL EXPRESSION ANALYSIS
# ============================================================================

print("\n🔬 PERFORMING DIFFERENTIAL EXPRESSION ANALYSIS...")

results = []

for i, gene in enumerate(gene_names):
    gene_expr = X_valid[:, i]
    
    # Check if this gene contains any target gene name
    gene_upper = gene.upper()
    is_target = any(target_gene in gene_upper for target_gene in target_genes)
    
    # Only analyze target genes with sufficient variability
    if is_target and len(np.unique(gene_expr)) > 5:
        # Split into high/low expression groups
        median_expr = np.median(gene_expr)
        high_expr = gene_expr > median_expr
        low_expr = gene_expr <= median_expr
        
        # Extract survival data for each group
        high_survival = survival_valid[high_expr]
        low_survival = survival_valid[low_expr]
        
        # Perform statistical test (Mann-Whitney U)
        if len(high_survival) > 10 and len(low_survival) > 10:
            from scipy.stats import mannwhitneyu
            stat, p_value = mannwhitneyu(high_survival, low_survival)
            
            # Calculate simple hazard ratio
            median_high = np.median(high_survival)
            median_low = np.median(low_survival)
            
            if median_low > 0:
                hr = median_high / median_low
            else:
                hr = np.nan
            
            # Find which target gene this corresponds to
            matched_target = next((tg for tg in target_genes if tg in gene_upper), 'Other')
            
            results.append({
                'Gene': gene,
                'Target_Gene': matched_target,
                'P_value': p_value,
                'HR': hr,
                'Median_High': median_high,
                'Median_Low': median_low,
                'High_Expr_Samples': np.sum(high_expr),
                'Low_Expr_Samples': np.sum(low_expr)
            })

# Convert to DataFrame and sort by p-value
results_df = pd.DataFrame(results).sort_values('P_value')

print(f"✅ Analysis completed:")
print(f"   • Genes analyzed: {len(results)}")
print(f"   • Significant genes (p < 0.05): {np.sum(results_df['P_value'] < 0.05)}")
print(f"   • Significant genes (p < 0.01): {np.sum(results_df['P_value'] < 0.01)}")
print("="*70)


🔬 PERFORMING DIFFERENTIAL EXPRESSION ANALYSIS...
✅ Analysis completed:
   • Genes analyzed: 40
   • Significant genes (p < 0.05): 3
   • Significant genes (p < 0.01): 0


In [7]:
# ============================================================================
# CELL 5: DISPLAY TOP RESULTS
# ============================================================================

print("\n🏆 TOP SURVIVAL-ASSOCIATED GENES:")
print("-"*70)

if len(results_df) > 0:
    for i, row in results_df.head(15).iterrows():
        # Determine significance markers
        if row['P_value'] < 0.01:
            significance = "**"
        elif row['P_value'] < 0.05:
            significance = "*"
        else:
            significance = ""
        
        # Determine direction of effect
        if row['HR'] > 1.1:
            direction = "Better"
        elif row['HR'] < 0.9:
            direction = "Worse"
        else:
            direction = "Neutral"
        
        print(f"{i+1:2d}. {row['Gene']:20s} [{row['Target_Gene']:8s}] : "
              f"p={row['P_value']:.4f}{significance}, "
              f"HR={row['HR']:.2f} ({direction} survival with high expression)")
else:
    print("No significant results found")

print("-"*70)


🏆 TOP SURVIVAL-ASSOCIATED GENES:
----------------------------------------------------------------------
26. RPGRIP1              [PGR     ] : p=0.0240*, HR=0.91 (Neutral survival with high expression)
25. RPGRIP1L             [PGR     ] : p=0.0424*, HR=1.20 (Better survival with high expression)
37. TP53TG1              [TP53    ] : p=0.0473*, HR=0.82 (Worse survival with high expression)
34. TP53INP1             [TP53    ] : p=0.1176, HR=0.95 (Neutral survival with high expression)
17. MYCT1                [MYC     ] : p=0.1317, HR=1.10 (Better survival with high expression)
40. TP53                 [TP53    ] : p=0.1579, HR=1.04 (Neutral survival with high expression)
 9. ESR1                 [ESR1    ] : p=0.1946, HR=1.11 (Better survival with high expression)
20. PGRMC2               [PGR     ] : p=0.1960, HR=0.89 (Worse survival with high expression)
14. MYCL1                [MYC     ] : p=0.1978, HR=1.14 (Better survival with high expression)
13. MYCBP                [MYC     ] 

In [9]:
# ============================================================================
# CELL 6: CREATE VISUALIZATIONS
# ============================================================================

print("\n🎨 CREATING VISUALIZATIONS...")
import matplotlib.pyplot as plt  # Add this import
import os
# Create output directory
os.makedirs('./biomarker_results', exist_ok=True)

if len(results_df) > 0:
    # Select top 6 genes for visualization
    top_genes = results_df.head(6)
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    for idx, (_, row) in enumerate(top_genes.iterrows()):
        if idx < len(axes):
            ax = axes[idx]
            
            # Find gene index
            gene_idx = next((i for i, g in enumerate(gene_names) 
                           if row['Gene'] in g), None)
            
            if gene_idx is not None:
                gene_expr = X_valid[:, gene_idx]
                median_expr = np.median(gene_expr)
                
                # Split groups
                high_expr_group = survival_valid[gene_expr > median_expr]
                low_expr_group = survival_valid[gene_expr <= median_expr]
                
                # Create box plot
                data_to_plot = [high_expr_group, low_expr_group]
                box = ax.boxplot(data_to_plot, patch_artist=True, 
                                labels=['High Expression', 'Low Expression'])
                
                # Color the boxes
                colors = ['lightgreen', 'lightcoral']
                for patch, color in zip(box['boxes'], colors):
                    patch.set_facecolor(color)
                
                # Add title with statistics
                title = f"{row['Gene']}\n"
                if row['P_value'] < 0.05:
                    title += f"p={row['P_value']:.4f}*, HR={row['HR']:.2f}"
                else:
                    title += f"p={row['P_value']:.4f}, HR={row['HR']:.2f}"
                
                ax.set_title(title, fontsize=11, fontweight='bold')
                ax.set_ylabel('Survival Time (days)', fontsize=10)
                ax.grid(True, alpha=0.3)
                ax.tick_params(axis='x', rotation=45)
    
    plt.suptitle('Survival Analysis of Key Breast Cancer Biomarkers', 
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('./biomarker_results/survival_biomarkers.png', 
                dpi=300, bbox_inches='tight')
    plt.close()
    
    print("✅ Biomarker visualization saved: './biomarker_results/survival_biomarkers.png'")
else:
    print("⚠️ No significant biomarkers to visualize")

print("="*70)


🎨 CREATING VISUALIZATIONS...
✅ Biomarker visualization saved: './biomarker_results/survival_biomarkers.png'


In [10]:
# ============================================================================
# CELL 7: GENERATE STATISTICAL SUMMARY
# ============================================================================

print("\n📊 GENERATING STATISTICAL SUMMARY...")

# Create summary statistics
summary_stats = {
    'total_samples': X_valid.shape[0],
    'median_survival': median_survival,
    'genes_analyzed': len(results),
    'significant_p_05': np.sum(results_df['P_value'] < 0.05),
    'significant_p_01': np.sum(results_df['P_value'] < 0.01),
    'top_genes': []
}

if len(results_df) > 0:
    for i, row in results_df.head(5).iterrows():
        summary_stats['top_genes'].append({
            'gene': row['Gene'],
            'target': row['Target_Gene'],
            'p_value': row['P_value'],
            'hr': row['HR'],
            'samples_high': row['High_Expr_Samples'],
            'samples_low': row['Low_Expr_Samples']
        })

# Save summary to file
summary_df = pd.DataFrame(summary_stats['top_genes'])
summary_df.to_csv('./biomarker_results/summary_statistics.csv', index=False)

print("📋 SUMMARY STATISTICS:")
print("-"*50)
print(f"Total Samples Analyzed: {summary_stats['total_samples']}")
print(f"Median Survival Time: {summary_stats['median_survival']:.1f} days")
print(f"Genes Analyzed: {summary_stats['genes_analyzed']}")
print(f"Significant Genes (p < 0.05): {summary_stats['significant_p_05']}")
print(f"Significant Genes (p < 0.01): {summary_stats['significant_p_01']}")
print("\nTop 5 Biomarkers:")
for i, gene_info in enumerate(summary_stats['top_genes'], 1):
    print(f"  {i}. {gene_info['gene']} ({gene_info['target']}): "
          f"p={gene_info['p_value']:.4f}, HR={gene_info['hr']:.2f}")
print("-"*50)
print("✅ Summary statistics saved: './biomarker_results/summary_statistics.csv'")
print("="*70)


📊 GENERATING STATISTICAL SUMMARY...
📋 SUMMARY STATISTICS:
--------------------------------------------------
Total Samples Analyzed: 1093
Median Survival Time: 786.0 days
Genes Analyzed: 40
Significant Genes (p < 0.05): 3
Significant Genes (p < 0.01): 0

Top 5 Biomarkers:
  1. RPGRIP1 (PGR): p=0.0240, HR=0.91
  2. RPGRIP1L (PGR): p=0.0424, HR=1.20
  3. TP53TG1 (TP53): p=0.0473, HR=0.82
  4. TP53INP1 (TP53): p=0.1176, HR=0.95
  5. MYCT1 (MYC): p=0.1317, HR=1.10
--------------------------------------------------
✅ Summary statistics saved: './biomarker_results/summary_statistics.csv'


In [11]:
# ============================================================================
# CELL 8: CREATE FOREST PLOT FOR SIGNIFICANT GENES
# ============================================================================

print("\n📈 CREATING FOREST PLOT FOR SIGNIFICANT BIOMARKERS...")

if len(results_df) > 0:
    # Filter significant genes (p < 0.05)
    significant_genes = results_df[results_df['P_value'] < 0.05].copy()
    
    if len(significant_genes) > 0:
        # Create forest plot
        plt.figure(figsize=(10, 6))
        
        # Prepare data
        genes = significant_genes['Gene'].values
        hr_values = significant_genes['HR'].values
        p_values = significant_genes['P_value'].values
        
        # Create positions
        y_pos = np.arange(len(genes))
        
        # Plot HR points
        plt.scatter(hr_values, y_pos, color='red', s=100, zorder=5, label='Hazard Ratio')
        
        # Add reference line at HR=1
        plt.axvline(x=1, color='black', linestyle='--', alpha=0.5, label='No Effect (HR=1)')
        
        # Add error bars (simplified)
        for i, (hr, p) in enumerate(zip(hr_values, p_values)):
            # Simple confidence interval approximation
            ci_width = 0.1 * (1 - p)  # Wider for less significant
            plt.plot([hr - ci_width, hr + ci_width], [i, i], 
                    color='blue', linewidth=2)
        
        # Customize plot
        plt.yticks(y_pos, genes)
        plt.xlabel('Hazard Ratio (HR)', fontsize=12)
        plt.title('Forest Plot of Significant Survival Biomarkers\n'
                 'Breast Cancer - TCGA-BRCA Cohort', 
                 fontsize=14, fontweight='bold')
        plt.grid(True, alpha=0.3, axis='x')
        plt.legend()
        
        # Add significance annotation
        for i, p in enumerate(p_values):
            if p < 0.05:
                plt.text(hr_values[i] + 0.05, i, '*', 
                        fontsize=12, fontweight='bold', va='center')
        
        plt.tight_layout()
        plt.savefig('./biomarker_results/forest_plot.png', 
                   dpi=300, bbox_inches='tight')
        plt.close()
        
        print("✅ Forest plot saved: './biomarker_results/forest_plot.png'")
    else:
        print("⚠️ No significant genes (p < 0.05) for forest plot")
else:
    print("⚠️ No results available for forest plot")

print("="*70)


📈 CREATING FOREST PLOT FOR SIGNIFICANT BIOMARKERS...
✅ Forest plot saved: './biomarker_results/forest_plot.png'


In [13]:
# ============================================================================
# CELL 9: SAVE COMPLETE RESULTS
# ============================================================================

print("\n💾 SAVING COMPLETE ANALYSIS RESULTS...")

import pandas as pd
import pickle
import numpy as np

# First, let's check what variables are available
available_vars = []
for var_name in ['X_gene', 'X_valid', 'results_df', 'target_genes', 'median_survival', 'summary_stats']:
    if var_name in locals() or var_name in globals():
        available_vars.append(var_name)

print(f"   Available variables: {available_vars}")

# Prepare complete results with error handling
complete_results = {
    'analysis_date': pd.Timestamp.now().strftime('%Y-%m-%d'),
    'cohort_info': {},
    'target_genes': [],
    'all_results': [],
    'significant_results': [],
    'summary_statistics': {}
}

try:
    # Try to get cohort info
    if 'X_gene' in locals() or 'X_gene' in globals():
        complete_results['cohort_info']['total_samples'] = X_gene.shape[0]
    elif 'X_valid' in locals() or 'X_valid' in globals():
        complete_results['cohort_info']['valid_samples'] = X_valid.shape[0]
    
    if 'median_survival' in locals() or 'median_survival' in globals():
        complete_results['cohort_info']['median_survival'] = median_survival
    
    # Get target genes
    if 'target_genes' in locals() or 'target_genes' in globals():
        complete_results['target_genes'] = target_genes
    
    # Get results
    if 'results_df' in locals() or 'results_df' in globals():
        complete_results['all_results'] = results_df.to_dict('records')
        significant_df = results_df[results_df['P_value'] < 0.05]
        complete_results['significant_results'] = significant_df.to_dict('records')
    
    # Get summary stats
    if 'summary_stats' in locals() or 'summary_stats' in globals():
        complete_results['summary_statistics'] = summary_stats
    
except Exception as e:
    print(f"   ⚠️  Warning: {e}")
    print("   Creating minimal results structure...")

# Create directory if it doesn't exist
import os
os.makedirs('./biomarker_results', exist_ok=True)

# Save to pickle
with open('./biomarker_results/complete_analysis_results.pkl', 'wb') as f:
    pickle.dump(complete_results, f)

# Also save results to CSV if available
if 'results_df' in locals() or 'results_df' in globals():
    results_df.to_csv('./biomarker_results/complete_gene_analysis.csv', index=False)
    print("   ✅ Gene analysis results saved to CSV")

print("📁 FILES SAVED:")
print("  1. './biomarker_results/complete_analysis_results.pkl' - Complete analysis results")

# List all files in biomarker_results
if os.path.exists('./biomarker_results'):
    files = os.listdir('./biomarker_results')
    for i, file in enumerate(files, 2):
        print(f"  {i}. './biomarker_results/{file}'")

print("\n✅ All results saved successfully!")
print("="*70)


💾 SAVING COMPLETE ANALYSIS RESULTS...
   Available variables: ['X_gene', 'X_valid', 'results_df', 'target_genes', 'median_survival', 'summary_stats']
   ✅ Gene analysis results saved to CSV
📁 FILES SAVED:
  1. './biomarker_results/complete_analysis_results.pkl' - Complete analysis results
  2. './biomarker_results/biomarker_report.txt'
  3. './biomarker_results/complete_analysis_results.pkl'
  4. './biomarker_results/complete_gene_analysis.csv'
  5. './biomarker_results/forest_plot.png'
  6. './biomarker_results/summary_statistics.csv'
  7. './biomarker_results/survival_biomarkers.png'

✅ All results saved successfully!
